In [ ]:
# ============================================================
# FIGURE 2(b) — V5 STRICTLY RDF-DRIVEN
# Verified RDF knowledge-graph excerpt
# RDFLib + NetworkX + PyGraphviz / Graphviz (FDP)
#
# PRINCIPLE:
# - No scientific RDF relation is invented or hard-coded as present.
# - The script reads the TTL, selects two verified operational devices,
#   discovers their actual semantic points and representative observations,
#   then copies ONLY triples explicitly present in the RDF graph.
# - Styling and abbreviated labels affect visualization only.
# ============================================================

from pathlib import Path
from collections import defaultdict
import re
import networkx as nx
from rdflib import Graph, URIRef, Literal, RDF, Namespace

# ------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------

TTL_PATH = Path(r"outputs/03_rdf/taltech_native_semantic_graph.ttl")
OUTPUT_DIR = Path(r"outputs/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UUIDS = [
    "99ca72a1-36de-4e79-b65d-16cdec7e42fb",
    "c6f4d193-bab8-438a-8706-d2f60e0ef81b",
]

# We display one representative observation per point.
OBSERVATIONS_PER_POINT = 1

# Literal values/timestamps/provenance are deliberately omitted from this
# publication excerpt to avoid visual overload. Set True only if required.
INCLUDE_LITERAL_DETAILS = False

EX = Namespace("https://example.org/taltech-edt/")
BRICK = Namespace("https://brickschema.org/schema/Brick#")
SOSA = Namespace("http://www.w3.org/ns/sosa/")
PROV = Namespace("http://www.w3.org/ns/prov#")

# Predicates eligible for the publication excerpt.
# IMPORTANT: this is only a display FILTER.
# A relation is drawn only if the exact triple exists in the TTL.
DISPLAY_PREDICATES = {
    RDF.type,
    EX.representsMeasurementFrom,
    EX.observedBySemanticPoint,
    EX.sourceOperationalDevice,
    SOSA.observedProperty,
}

if INCLUDE_LITERAL_DETAILS:
    DISPLAY_PREDICATES |= {
        SOSA.hasSimpleResult,
        SOSA.resultTime,
        PROV.wasDerivedFrom,
    }

# ------------------------------------------------------------
# 1. DEPENDENCY CHECK
# ------------------------------------------------------------

try:
    import pygraphviz as pgv
except ImportError as e:
    raise RuntimeError(
        "PyGraphviz is not installed.\n"
        "Run in Anaconda Prompt:\n"
        "conda activate edt_these\n"
        "conda install -c conda-forge graphviz pygraphviz"
    ) from e

print("NetworkX:", nx.__version__)
print("PyGraphviz:", pgv.__version__)

if not TTL_PATH.exists():
    raise FileNotFoundError(
        f"TTL file not found:\n{TTL_PATH.resolve()}\n\n"
        "Adjust TTL_PATH at the beginning of the script."
    )

# ------------------------------------------------------------
# 2. LOAD THE ACTUAL TTL WITH RDFLIB
# ------------------------------------------------------------

print("\nLoading RDF graph...")
rdf = Graph()
rdf.parse(TTL_PATH, format="turtle")

print(f"✓ RDF loaded: {len(rdf):,} triples")

# ------------------------------------------------------------
# 3. PREFIX / LABEL HELPERS
# ------------------------------------------------------------

PREFIXES = [
    (str(BRICK), "brick:"),
    (str(SOSA), "sosa:"),
    (str(PROV), "prov:"),
    (str(EX), "ex:"),
    (str(RDF), "rdf:"),
]

def qname(term):
    """Readable QName-like label without changing the RDF term."""
    if isinstance(term, Literal):
        return str(term)

    s = str(term)
    for base, prefix in PREFIXES:
        if s.startswith(base):
            return prefix + s[len(base):]

    if "#" in s:
        return s.rsplit("#", 1)[-1]
    return s.rsplit("/", 1)[-1]

def short_uuid(uuid, n=8):
    return uuid[:n] + "…"

def short_observation(term, n=7):
    local = str(term).rsplit("/", 1)[-1]
    return local[:n] + "…"

# ------------------------------------------------------------
# 4. FIND THE TWO OPERATIONAL DEVICES FROM THE RDF ITSELF
# ------------------------------------------------------------

devices = {}

for uuid in UUIDS:
    candidates = [
        s for s in rdf.subjects(RDF.type, EX.OperationalDevice)
        if uuid in str(s)
    ]

    if len(candidates) != 1:
        raise ValueError(
            f"Expected exactly one ex:OperationalDevice for {uuid}; "
            f"found {len(candidates)}."
        )

    devices[uuid] = candidates[0]

print("\n✓ Operational devices verified:")
for uuid, uri in devices.items():
    print(" -", uuid, "->", uri)

# ------------------------------------------------------------
# 5. DISCOVER ACTUAL SEMANTIC POINTS
#
# We DO NOT assume ex:hasSemanticPoint.
# Points are discovered from the explicitly materialized inverse relation:
#     point --ex:representsMeasurementFrom--> device
# ------------------------------------------------------------

points_by_device = defaultdict(list)

for uuid, device in devices.items():
    actual_points = list(
        rdf.subjects(EX.representsMeasurementFrom, device)
    )

    if not actual_points:
        raise ValueError(
            f"No explicit ex:representsMeasurementFrom points found "
            f"for device {uuid}."
        )

    points_by_device[uuid] = actual_points

print("\n✓ Explicit semantic points discovered from TTL:")
for uuid, points in points_by_device.items():
    print(f" - {short_uuid(uuid)}: {len(points)} point(s)")
    for p in points:
        brick_types = [
            o for o in rdf.objects(p, RDF.type)
            if str(o).startswith(str(BRICK))
        ]
        print(
            "    ",
            qname(p),
            " | type:",
            ", ".join(qname(t) for t in brick_types) or "not shown"
        )

# ------------------------------------------------------------
# 6. SELECT ONE REAL SOSA OBSERVATION PER DISCOVERED POINT
#
# Observations are discovered ONLY through the explicit triple:
#     observation --ex:observedBySemanticPoint--> point
# ------------------------------------------------------------

observations_by_point = defaultdict(list)

for uuid, points in points_by_device.items():
    for point in points:
        candidates = list(
            rdf.subjects(EX.observedBySemanticPoint, point)
        )

        # Keep only actual SOSA observations.
        candidates = [
            obs for obs in candidates
            if (obs, RDF.type, SOSA.Observation) in rdf
        ]

        candidates = sorted(candidates, key=lambda x: str(x))

        if not candidates:
            print(
                "WARNING: no explicit SOSA observation found for",
                qname(point)
            )
            continue

        observations_by_point[point] = (
            candidates[:OBSERVATIONS_PER_POINT]
        )

print("\n✓ Representative observations selected:")
for point, observations in observations_by_point.items():
    for obs in observations:
        properties = list(
            rdf.objects(obs, SOSA.observedProperty)
        )
        print(
            " -",
            short_observation(obs),
            "->",
            qname(point),
            "| observedProperty:",
            ", ".join(qname(p) for p in properties) or "none"
        )

# ------------------------------------------------------------
# 7. DEFINE THE SELECTED RDF RESOURCES
# ------------------------------------------------------------

selected_resources = set()

for device in devices.values():
    selected_resources.add(device)

for points in points_by_device.values():
    selected_resources.update(points)

for observations in observations_by_point.values():
    selected_resources.update(observations)

# Add objects reached through eligible predicates.
# This brings in actual Brick classes, SOSA class and observed properties,
# but ONLY when such triples are explicitly present.
for subject in list(selected_resources):
    for predicate, obj in rdf.predicate_objects(subject):
        if predicate in DISPLAY_PREDICATES:
            if isinstance(obj, URIRef):
                selected_resources.add(obj)

# ------------------------------------------------------------
# 8. COPY ONLY EXPLICIT TTL TRIPLES INTO THE FIGURE SUBGRAPH
# ------------------------------------------------------------

selected_triples = []

for subject in selected_resources:
    for predicate, obj in rdf.predicate_objects(subject):

        if predicate not in DISPLAY_PREDICATES:
            continue

        if isinstance(obj, URIRef):
            # URI object must belong to the selected excerpt.
            if obj not in selected_resources:
                continue

        elif isinstance(obj, Literal):
            if not INCLUDE_LITERAL_DETAILS:
                continue

        selected_triples.append((subject, predicate, obj))

# Remove duplicates while preserving deterministic order.
selected_triples = sorted(
    set(selected_triples),
    key=lambda t: (str(t[0]), str(t[1]), str(t[2]))
)

print(
    f"\n✓ Figure subgraph contains "
    f"{len(selected_triples)} explicit TTL triples."
)

# Critical audit: prove that every displayed triple exists in rdf.
for triple in selected_triples:
    if triple not in rdf:
        raise AssertionError(
            f"NON-RDF TRIPLE DETECTED: {triple}"
        )

print("✓ Audit passed: every displayed relation exists in the TTL.")

# ------------------------------------------------------------
# 9. BUILD NETWORKX GRAPH FROM THOSE VERIFIED TRIPLES
# ------------------------------------------------------------

G = nx.MultiDiGraph()

device_terms = set(devices.values())
point_terms = {
    p for points in points_by_device.values() for p in points
}
observation_terms = {
    o for obs_list in observations_by_point.values()
    for o in obs_list
}

observed_property_terms = set()
brick_class_terms = set()

for s, p, o in selected_triples:
    if p == SOSA.observedProperty and isinstance(o, URIRef):
        observed_property_terms.add(o)

    if (
        p == RDF.type
        and isinstance(o, URIRef)
        and str(o).startswith(str(BRICK))
    ):
        brick_class_terms.add(o)

def node_type(term):
    if isinstance(term, Literal):
        return "literal"
    if term in device_terms:
        return "device"
    if term in point_terms:
        return "point"
    if term in observation_terms:
        return "observation"
    if term in observed_property_terms:
        return "property"
    return "class"

def node_label(term):
    typ = node_type(term)

    if typ == "device":
        matching_uuid = next(
            uuid for uuid, uri in devices.items()
            if uri == term
        )
        return (
            "Operational device\n"
            + short_uuid(matching_uuid)
        )

    if typ == "point":
        # Use the actual Brick type to make the visual label informative.
        brick_types = [
            o for o in rdf.objects(term, RDF.type)
            if str(o).startswith(str(BRICK))
        ]

        if brick_types:
            bt = qname(brick_types[0])
            replacements = {
                "brick:Temperature_Sensor": "Temperature point",
                "brick:Relative_Humidity_Sensor": "RH point",
                "brick:CO2_Sensor": "CO₂ point",
            }
            base = replacements.get(bt, qname(term))
        else:
            base = qname(term)

        # Device abbreviation is visual only.
        parent_devices = list(
            rdf.objects(term, EX.representsMeasurementFrom)
        )

        suffix = ""
        if parent_devices:
            parent = str(parent_devices[0])
            matching = [
                uuid for uuid in UUIDS
                if uuid in parent
            ]
            if matching:
                suffix = "\n" + matching[0][:4] + "…"

        return base + suffix

    if typ == "observation":
        return (
            "sosa:Observation\n"
            + short_observation(term)
        )

    if typ == "literal":
        return str(term)

    return qname(term)

for s, p, o in selected_triples:

    if s not in G:
        G.add_node(
            s,
            label=node_label(s),
            type=node_type(s)
        )

    if o not in G:
        G.add_node(
            o,
            label=node_label(o),
            type=node_type(o)
        )

    G.add_edge(
        s,
        o,
        predicate=qname(p),
        predicate_uri=str(p)
    )

# ------------------------------------------------------------
# 10. GRAPHVIZ STYLE
# ------------------------------------------------------------

A = pgv.AGraph(
    directed=True,
    strict=False
)

A.graph_attr.update(
    bgcolor="white",
    overlap="prism",
    splines="curved",
    outputorder="edgesfirst",
    pad="0.30",
    sep="+22",
    esep="+10",
    K="0.85",
    labelloc="t",
    label="(b) Excerpt of the instantiated RDF knowledge graph",
    fontsize="18",
    fontname="Arial"
)

A.node_attr.update(
    fontname="Arial",
    fontsize="10",
    margin="0.15,0.10",
    penwidth="1.4"
)

# Compact visual legend for secondary relations.
# These legend elements are GRAPHICAL ANNOTATIONS ONLY and are not included
# in the RDF audit / selected_triples.
A.add_node(
    "LEGEND",
    label=(
        "Edge styles\\l"
        "– –  rdf:type\\l"
        "···  ex:sourceOperationalDevice\\l"
    ),
    shape="box",
    style="rounded",
    color="#A0A0A0",
    fontcolor="#555555",
    fontsize="9",
    margin="0.12,0.08"
)

A.edge_attr.update(
    fontname="Arial",
    fontsize="9",
    arrowsize="0.72",
    penwidth="1.0"
)

NODE_STYLES = {
    "device": {
        "shape": "ellipse",
        "style": "filled",
        "fillcolor": "#DCEEFF",
        "color": "#2166AC",
    },
    "point": {
        "shape": "ellipse",
        "style": "filled",
        "fillcolor": "#E5F3E0",
        "color": "#238B45",
    },
    "observation": {
        "shape": "ellipse",
        "style": "filled",
        "fillcolor": "#EEE5F8",
        "color": "#7A3DB8",
    },
    "property": {
        "shape": "diamond",
        "style": "filled",
        "fillcolor": "#FCE5EA",
        "color": "#C51B7D",
    },
    "class": {
        "shape": "box",
        "style": "rounded,filled",
        "fillcolor": "#E8EEF5",
        "color": "#607D9B",
    },
    "literal": {
        "shape": "box",
        "style": "rounded,filled",
        "fillcolor": "#F7F7F7",
        "color": "#888888",
    },
}

# Styling is keyed by actual qname labels.
# It does not create any relation.
EDGE_STYLES = {
    "rdf:type": {
        "color": "#B8C2CC",
        "fontcolor": "#667788",
        "style": "dashed",
        "visual_label": "",
    },
    "ex:representsMeasurementFrom": {
        "color": "#D95F02",
        "fontcolor": "#D95F02",
        "style": "dashed",
        "visual_label": "representsMeasurementFrom",
    },
    "ex:observedBySemanticPoint": {
        "color": "#238B45",
        "fontcolor": "#238B45",
        "style": "solid",
        "visual_label": "observedBySemanticPoint",
    },
    "ex:sourceOperationalDevice": {
        "color": "#C8C8C8",
        "fontcolor": "#777777",
        "style": "dotted",
        # Relation remains in the graph; only repeated text is hidden.
        "visual_label": "",
    },
    "sosa:observedProperty": {
        "color": "#C51B7D",
        "fontcolor": "#C51B7D",
        "style": "solid",
        "visual_label": "observedProperty",
    },
    "sosa:hasSimpleResult": {
        "color": "#666666",
        "fontcolor": "#666666",
        "style": "solid",
        "visual_label": "hasSimpleResult",
    },
    "sosa:resultTime": {
        "color": "#666666",
        "fontcolor": "#666666",
        "style": "solid",
        "visual_label": "resultTime",
    },
    "prov:wasDerivedFrom": {
        "color": "#777777",
        "fontcolor": "#777777",
        "style": "dashed",
        "visual_label": "wasDerivedFrom",
    },
}

# ------------------------------------------------------------
# 11. ADD VERIFIED NODES AND EDGES TO GRAPHVIZ
# ------------------------------------------------------------

# PyGraphviz works most reliably with string IDs.
gv_id = {
    node: f"N{i}"
    for i, node in enumerate(G.nodes())
}

for node, data in G.nodes(data=True):

    style = NODE_STYLES[data["type"]]

    A.add_node(
        gv_id[node],
        label=data["label"],
        shape=style["shape"],
        style=style["style"],
        fillcolor=style["fillcolor"],
        color=style["color"],
        penwidth="1.4"
    )

for source, target, key, data in G.edges(
    keys=True,
    data=True
):
    predicate = data["predicate"]

    style = EDGE_STYLES.get(
        predicate,
        {
            "color": "#666666",
            "fontcolor": "#666666",
            "style": "solid",
            "visual_label": predicate,
        }
    )

    A.add_edge(
        gv_id[source],
        gv_id[target],
        label=style["visual_label"],
        color=style["color"],
        fontcolor=style["fontcolor"],
        style=style["style"],
        arrowsize="0.72",
        penwidth="1.0"
    )

# ------------------------------------------------------------
# 12. FORCE-DIRECTED LAYOUT
# ------------------------------------------------------------

# FDP keeps the knowledge-graph appearance and avoids DOT's tree layout.
A.layout(prog="fdp")

# ------------------------------------------------------------
# 13. EXPORT
# ------------------------------------------------------------

SVG_PATH = OUTPUT_DIR / "Figure_2b_RDF_V6_clean.svg"
PDF_PATH = OUTPUT_DIR / "Figure_2b_RDF_V6_clean.pdf"
PNG_PATH = OUTPUT_DIR / "Figure_2b_RDF_V6_clean.png"

A.draw(str(SVG_PATH), format="svg")
A.draw(str(PDF_PATH), format="pdf")
A.draw(str(PNG_PATH), format="png")

# ------------------------------------------------------------
# 14. REPRODUCIBILITY AUDIT FILE
# ------------------------------------------------------------

AUDIT_PATH = OUTPUT_DIR / "Figure_2b_RDF_V6_displayed_triples.tsv"

with AUDIT_PATH.open("w", encoding="utf-8") as f:
    f.write("subject\tpredicate\tobject\n")
    for s, p, o in selected_triples:
        f.write(
            f"{str(s)}\t{str(p)}\t{str(o)}\n"
        )

print("\n==============================================")
print("FIGURE 2(b) V6 COMPLETED")
print("==============================================")
print(f"Displayed nodes   : {G.number_of_nodes()}")
print(f"Displayed triples : {G.number_of_edges()}")
print("\nFiles:")
print(" SVG :", SVG_PATH.resolve())
print(" PDF :", PDF_PATH.resolve())
print(" PNG :", PNG_PATH.resolve())
print(" AUDIT:", AUDIT_PATH.resolve())

print(
    "\nScientific guarantee: every displayed edge was copied "
    "from an explicit triple in the loaded TTL. "
    "No ex:hasSemanticPoint relation is assumed or generated."
)


In [ ]:
# ============================================================
# FIGURE 2(b) — V5 STRICTLY RDF-DRIVEN
# Verified RDF knowledge-graph excerpt
# RDFLib + NetworkX + PyGraphviz / Graphviz (FDP)
#
# PRINCIPLE:
# - No scientific RDF relation is invented or hard-coded as present.
# - The script reads the TTL, selects two verified operational devices,
#   discovers their actual semantic points and representative observations,
#   then copies ONLY triples explicitly present in the RDF graph.
# - Styling and abbreviated labels affect visualization only.
# ============================================================

from pathlib import Path
from collections import defaultdict
import re
import networkx as nx
from rdflib import Graph, URIRef, Literal, RDF, Namespace

# ------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------

TTL_PATH = Path(r"outputs/03_rdf/taltech_native_semantic_graph.ttl")
OUTPUT_DIR = Path(r"outputs/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UUIDS = [
    "99ca72a1-36de-4e79-b65d-16cdec7e42fb",
    "c6f4d193-bab8-438a-8706-d2f60e0ef81b",
]

# We display one representative observation per point.
OBSERVATIONS_PER_POINT = 1

# Literal values/timestamps/provenance are deliberately omitted from this
# publication excerpt to avoid visual overload. Set True only if required.
INCLUDE_LITERAL_DETAILS = False

EX = Namespace("https://example.org/taltech-edt/")
BRICK = Namespace("https://brickschema.org/schema/Brick#")
SOSA = Namespace("http://www.w3.org/ns/sosa/")
PROV = Namespace("http://www.w3.org/ns/prov#")

# Predicates eligible for the publication excerpt.
# IMPORTANT: this is only a display FILTER.
# A relation is drawn only if the exact triple exists in the TTL.
DISPLAY_PREDICATES = {
    RDF.type,
    EX.representsMeasurementFrom,
    EX.observedBySemanticPoint,
    EX.sourceOperationalDevice,
    SOSA.observedProperty,
}

if INCLUDE_LITERAL_DETAILS:
    DISPLAY_PREDICATES |= {
        SOSA.hasSimpleResult,
        SOSA.resultTime,
        PROV.wasDerivedFrom,
    }

# ------------------------------------------------------------
# 1. DEPENDENCY CHECK
# ------------------------------------------------------------

try:
    import pygraphviz as pgv
except ImportError as e:
    raise RuntimeError(
        "PyGraphviz is not installed.\n"
        "Run in Anaconda Prompt:\n"
        "conda activate edt_these\n"
        "conda install -c conda-forge graphviz pygraphviz"
    ) from e

print("NetworkX:", nx.__version__)
print("PyGraphviz:", pgv.__version__)

if not TTL_PATH.exists():
    raise FileNotFoundError(
        f"TTL file not found:\n{TTL_PATH.resolve()}\n\n"
        "Adjust TTL_PATH at the beginning of the script."
    )

# ------------------------------------------------------------
# 2. LOAD THE ACTUAL TTL WITH RDFLIB
# ------------------------------------------------------------

print("\nLoading RDF graph...")
rdf = Graph()
rdf.parse(TTL_PATH, format="turtle")

print(f"✓ RDF loaded: {len(rdf):,} triples")

# ------------------------------------------------------------
# 3. PREFIX / LABEL HELPERS
# ------------------------------------------------------------

PREFIXES = [
    (str(BRICK), "brick:"),
    (str(SOSA), "sosa:"),
    (str(PROV), "prov:"),
    (str(EX), "ex:"),
    (str(RDF), "rdf:"),
]

def qname(term):
    """Readable QName-like label without changing the RDF term."""
    if isinstance(term, Literal):
        return str(term)

    s = str(term)
    for base, prefix in PREFIXES:
        if s.startswith(base):
            return prefix + s[len(base):]

    if "#" in s:
        return s.rsplit("#", 1)[-1]
    return s.rsplit("/", 1)[-1]

def short_uuid(uuid, n=8):
    return uuid[:n] + "…"

def short_observation(term, n=7):
    local = str(term).rsplit("/", 1)[-1]
    return local[:n] + "…"

# ------------------------------------------------------------
# 4. FIND THE TWO OPERATIONAL DEVICES FROM THE RDF ITSELF
# ------------------------------------------------------------

devices = {}

for uuid in UUIDS:
    candidates = [
        s for s in rdf.subjects(RDF.type, EX.OperationalDevice)
        if uuid in str(s)
    ]

    if len(candidates) != 1:
        raise ValueError(
            f"Expected exactly one ex:OperationalDevice for {uuid}; "
            f"found {len(candidates)}."
        )

    devices[uuid] = candidates[0]

print("\n✓ Operational devices verified:")
for uuid, uri in devices.items():
    print(" -", uuid, "->", uri)

# ------------------------------------------------------------
# 5. DISCOVER ACTUAL SEMANTIC POINTS
#
# We DO NOT assume ex:hasSemanticPoint.
# Points are discovered from the explicitly materialized inverse relation:
#     point --ex:representsMeasurementFrom--> device
# ------------------------------------------------------------

points_by_device = defaultdict(list)

for uuid, device in devices.items():
    actual_points = list(
        rdf.subjects(EX.representsMeasurementFrom, device)
    )

    if not actual_points:
        raise ValueError(
            f"No explicit ex:representsMeasurementFrom points found "
            f"for device {uuid}."
        )

    points_by_device[uuid] = actual_points

print("\n✓ Explicit semantic points discovered from TTL:")
for uuid, points in points_by_device.items():
    print(f" - {short_uuid(uuid)}: {len(points)} point(s)")
    for p in points:
        brick_types = [
            o for o in rdf.objects(p, RDF.type)
            if str(o).startswith(str(BRICK))
        ]
        print(
            "    ",
            qname(p),
            " | type:",
            ", ".join(qname(t) for t in brick_types) or "not shown"
        )

# ------------------------------------------------------------
# 6. SELECT ONE REAL SOSA OBSERVATION PER DISCOVERED POINT
#
# Observations are discovered ONLY through the explicit triple:
#     observation --ex:observedBySemanticPoint--> point
# ------------------------------------------------------------

observations_by_point = defaultdict(list)

for uuid, points in points_by_device.items():
    for point in points:
        candidates = list(
            rdf.subjects(EX.observedBySemanticPoint, point)
        )

        # Keep only actual SOSA observations.
        candidates = [
            obs for obs in candidates
            if (obs, RDF.type, SOSA.Observation) in rdf
        ]

        candidates = sorted(candidates, key=lambda x: str(x))

        if not candidates:
            print(
                "WARNING: no explicit SOSA observation found for",
                qname(point)
            )
            continue

        observations_by_point[point] = (
            candidates[:OBSERVATIONS_PER_POINT]
        )

print("\n✓ Representative observations selected:")
for point, observations in observations_by_point.items():
    for obs in observations:
        properties = list(
            rdf.objects(obs, SOSA.observedProperty)
        )
        print(
            " -",
            short_observation(obs),
            "->",
            qname(point),
            "| observedProperty:",
            ", ".join(qname(p) for p in properties) or "none"
        )

# ------------------------------------------------------------
# 7. DEFINE THE SELECTED RDF RESOURCES
# ------------------------------------------------------------

selected_resources = set()

for device in devices.values():
    selected_resources.add(device)

for points in points_by_device.values():
    selected_resources.update(points)

for observations in observations_by_point.values():
    selected_resources.update(observations)

# Add objects reached through eligible predicates.
# This brings in actual Brick classes, SOSA class and observed properties,
# but ONLY when such triples are explicitly present.
for subject in list(selected_resources):
    for predicate, obj in rdf.predicate_objects(subject):
        if predicate in DISPLAY_PREDICATES:
            if isinstance(obj, URIRef):
                selected_resources.add(obj)

# ------------------------------------------------------------
# 8. COPY ONLY EXPLICIT TTL TRIPLES INTO THE FIGURE SUBGRAPH
# ------------------------------------------------------------

selected_triples = []

for subject in selected_resources:
    for predicate, obj in rdf.predicate_objects(subject):

        if predicate not in DISPLAY_PREDICATES:
            continue

        if isinstance(obj, URIRef):
            # URI object must belong to the selected excerpt.
            if obj not in selected_resources:
                continue

        elif isinstance(obj, Literal):
            if not INCLUDE_LITERAL_DETAILS:
                continue

        selected_triples.append((subject, predicate, obj))

# Remove duplicates while preserving deterministic order.
selected_triples = sorted(
    set(selected_triples),
    key=lambda t: (str(t[0]), str(t[1]), str(t[2]))
)

print(
    f"\n✓ Figure subgraph contains "
    f"{len(selected_triples)} explicit TTL triples."
)

# Critical audit: prove that every displayed triple exists in rdf.
for triple in selected_triples:
    if triple not in rdf:
        raise AssertionError(
            f"NON-RDF TRIPLE DETECTED: {triple}"
        )

print("✓ Audit passed: every displayed relation exists in the TTL.")

# ------------------------------------------------------------
# 9. BUILD NETWORKX GRAPH FROM THOSE VERIFIED TRIPLES
# ------------------------------------------------------------

G = nx.MultiDiGraph()

device_terms = set(devices.values())
point_terms = {
    p for points in points_by_device.values() for p in points
}
observation_terms = {
    o for obs_list in observations_by_point.values()
    for o in obs_list
}

observed_property_terms = set()
brick_class_terms = set()

for s, p, o in selected_triples:
    if p == SOSA.observedProperty and isinstance(o, URIRef):
        observed_property_terms.add(o)

    if (
        p == RDF.type
        and isinstance(o, URIRef)
        and str(o).startswith(str(BRICK))
    ):
        brick_class_terms.add(o)

def node_type(term):
    if isinstance(term, Literal):
        return "literal"
    if term in device_terms:
        return "device"
    if term in point_terms:
        return "point"
    if term in observation_terms:
        return "observation"
    if term in observed_property_terms:
        return "property"
    return "class"

def node_label(term):
    typ = node_type(term)

    if typ == "device":
        matching_uuid = next(
            uuid for uuid, uri in devices.items()
            if uri == term
        )
        return (
            "Operational device\n"
            + short_uuid(matching_uuid)
        )

    if typ == "point":
        # Use the actual Brick type to make the visual label informative.
        brick_types = [
            o for o in rdf.objects(term, RDF.type)
            if str(o).startswith(str(BRICK))
        ]

        if brick_types:
            bt = qname(brick_types[0])
            replacements = {
                "brick:Temperature_Sensor": "Temperature point",
                "brick:Relative_Humidity_Sensor": "RH point",
                "brick:CO2_Sensor": "CO₂ point",
            }
            base = replacements.get(bt, qname(term))
        else:
            base = qname(term)

        # Device abbreviation is visual only.
        parent_devices = list(
            rdf.objects(term, EX.representsMeasurementFrom)
        )

        suffix = ""
        if parent_devices:
            parent = str(parent_devices[0])
            matching = [
                uuid for uuid in UUIDS
                if uuid in parent
            ]
            if matching:
                suffix = "\n" + matching[0][:4] + "…"

        return base + suffix

    if typ == "observation":
        return (
            "sosa:Observation\n"
            + short_observation(term)
        )

    if typ == "literal":
        return str(term)

    return qname(term)

for s, p, o in selected_triples:

    if s not in G:
        G.add_node(
            s,
            label=node_label(s),
            type=node_type(s)
        )

    if o not in G:
        G.add_node(
            o,
            label=node_label(o),
            type=node_type(o)
        )

    G.add_edge(
        s,
        o,
        predicate=qname(p),
        predicate_uri=str(p)
    )

# ------------------------------------------------------------
# 10. GRAPHVIZ STYLE
# ------------------------------------------------------------

A = pgv.AGraph(
    directed=True,
    strict=False
)

A.graph_attr.update(
    bgcolor="white",
    overlap="prism",
    splines="curved",
    outputorder="edgesfirst",
    pad="0.18",
    sep="+16",
    esep="+8",
    K="0.68",
    pack="true",
    packmode="node",
    labelloc="t",
    label="(b)",
    fontsize="18",
    fontname="Arial"
)

A.node_attr.update(
    fontname="Arial",
    fontsize="10",
    margin="0.15,0.10",
    penwidth="1.4"
)

# Compact visual legend for secondary relations.
# These legend elements are GRAPHICAL ANNOTATIONS ONLY and are not included
# in the RDF audit / selected_triples.
A.add_node(
    "LEGEND",
    label="""<
<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="2" CELLPADDING="1">
<TR><TD COLSPAN="2" ALIGN="LEFT"><B>Edge relations</B></TD></TR>
<TR><TD><FONT COLOR="#238B45">-----</FONT></TD><TD ALIGN="LEFT"><FONT COLOR="#238B45">ex:observedBySemanticPoint</FONT></TD></TR>
<TR><TD><FONT COLOR="#C51B7D">-----</FONT></TD><TD ALIGN="LEFT"><FONT COLOR="#C51B7D">sosa:observedProperty</FONT></TD></TR>
<TR><TD><FONT COLOR="#D95F02">-----</FONT></TD><TD ALIGN="LEFT"><FONT COLOR="#D95F02">ex:representsMeasurementFrom</FONT></TD></TR>
<TR><TD><FONT COLOR="#AEBCC8">- - -</FONT></TD><TD ALIGN="LEFT"><FONT COLOR="#7E8D99">rdf:type</FONT></TD></TR>
<TR><TD><FONT COLOR="#A8A8A8">.....</FONT></TD><TD ALIGN="LEFT"><FONT COLOR="#777777">ex:sourceOperationalDevice</FONT></TD></TR>
</TABLE>
>""",
    shape="box",
    style="rounded",
    color="#A0A0A0",
    fontsize="11",
    margin="0.12,0.08",
    penwidth="1.1"
)

A.edge_attr.update(
    fontname="Arial",
    fontsize="11",
    arrowsize="0.78",
    penwidth="1.1"
)

NODE_STYLES = {
    "device": {
        "shape": "ellipse",
        "style": "filled",
        "fillcolor": "#DCEEFF",
        "color": "#2166AC",
    },
    "point": {
        "shape": "ellipse",
        "style": "filled",
        "fillcolor": "#E5F3E0",
        "color": "#238B45",
    },
    "observation": {
        "shape": "ellipse",
        "style": "filled",
        "fillcolor": "#EEE5F8",
        "color": "#7A3DB8",
    },
    "property": {
        "shape": "diamond",
        "style": "filled",
        "fillcolor": "#FCE5EA",
        "color": "#C51B7D",
    },
    "class": {
        "shape": "box",
        "style": "rounded,filled",
        "fillcolor": "#E8EEF5",
        "color": "#607D9B",
    },
    "literal": {
        "shape": "box",
        "style": "rounded,filled",
        "fillcolor": "#F7F7F7",
        "color": "#888888",
    },
}

# Styling is keyed by actual qname labels.
# It does not create any relation.
EDGE_STYLES = {
    "rdf:type": {
        "color": "#D1D9E0",
        "fontcolor": "#8A96A3",
        "style": "dashed",
        "visual_label": "",
        "penwidth": "0.75",
    },
    "ex:representsMeasurementFrom": {
        "color": "#D95F02",
        "fontcolor": "#D95F02",
        "style": "solid",
        "visual_label": "",
        "penwidth": "0.85",
    },
    "ex:observedBySemanticPoint": {
        "color": "#238B45",
        "fontcolor": "#238B45",
        "style": "solid",
        "visual_label": "observedBySemanticPoint",
        "penwidth": "1.15",
    },
    "ex:sourceOperationalDevice": {
        "color": "#C8C8C8",
        "fontcolor": "#777777",
        "style": "dotted",
        "visual_label": "",
        "penwidth": "0.70",
    },
    "sosa:observedProperty": {
        "color": "#C51B7D",
        "fontcolor": "#C51B7D",
        "style": "solid",
        "visual_label": "observedProperty",
        "penwidth": "1.15",
    },
    "sosa:hasSimpleResult": {
        "color": "#666666",
        "fontcolor": "#666666",
        "style": "solid",
        "visual_label": "hasSimpleResult",
    },
    "sosa:resultTime": {
        "color": "#666666",
        "fontcolor": "#666666",
        "style": "solid",
        "visual_label": "resultTime",
    },
    "prov:wasDerivedFrom": {
        "color": "#777777",
        "fontcolor": "#777777",
        "style": "dashed",
        "visual_label": "wasDerivedFrom",
    },
}

# ------------------------------------------------------------
# 11. ADD VERIFIED NODES AND EDGES TO GRAPHVIZ
# ------------------------------------------------------------

# PyGraphviz works most reliably with string IDs.
gv_id = {
    node: f"N{i}"
    for i, node in enumerate(G.nodes())
}

for node, data in G.nodes(data=True):

    style = NODE_STYLES[data["type"]]

    A.add_node(
        gv_id[node],
        label=data["label"],
        shape=style["shape"],
        style=style["style"],
        fillcolor=style["fillcolor"],
        color=style["color"],
        penwidth="1.4"
    )

for source, target, key, data in G.edges(
    keys=True,
    data=True
):
    predicate = data["predicate"]

    style = EDGE_STYLES.get(
        predicate,
        {
            "color": "#666666",
            "fontcolor": "#666666",
            "style": "solid",
            "visual_label": predicate,
        }
    )

    A.add_edge(
        gv_id[source],
        gv_id[target],
        label=style["visual_label"],
        color=style["color"],
        fontcolor=style["fontcolor"],
        style=style["style"],
        arrowsize="0.78",
        penwidth=style.get("penwidth", "1.0"),
        fontsize="11"
    )

# ------------------------------------------------------------
# 12. V9 SEMI-MANUAL PUBLICATION LAYOUT
#
# RDF content is unchanged. Only coordinates are modified to:
# - separate observations from shared properties,
# - avoid label/node overlap,
# - move ontology classes to the perimeter,
# - preserve a symmetric network appearance.
# ------------------------------------------------------------

device_nodes = [n for n, d in G.nodes(data=True) if d["type"] == "device"]
point_nodes = [n for n, d in G.nodes(data=True) if d["type"] == "point"]
obs_nodes = [n for n, d in G.nodes(data=True) if d["type"] == "observation"]
property_nodes = [n for n, d in G.nodes(data=True) if d["type"] == "property"]
class_nodes = [n for n, d in G.nodes(data=True) if d["type"] == "class"]

def linked_device(point):
    vals = list(rdf.objects(point, EX.representsMeasurementFrom))
    return vals[0] if vals else None

def point_brick_type(point):
    vals = [
        o for o in rdf.objects(point, RDF.type)
        if str(o).startswith(str(BRICK))
    ]
    return vals[0] if vals else None

def obs_point(obs):
    vals = list(rdf.objects(obs, EX.observedBySemanticPoint))
    return vals[0] if vals else None

# Order the two real devices deterministically.
device_order = [
    devices[uuid] for uuid in UUIDS
    if devices[uuid] in G
]

POS = {}

# Devices farther apart horizontally.
if len(device_order) >= 1:
    POS[device_order[0]] = (-560, 0)
if len(device_order) >= 2:
    POS[device_order[1]] = (560, 0)

# Shared observed properties remain central.
property_y = {
    "ex:IndoorTemperature": 330,
    "ex:RelativeHumidity": 0,
    "ex:CO2Concentration": -330,
}
for n in property_nodes:
    POS[n] = (0, property_y.get(qname(n), 0))

# Points: close to their own device but aligned by variable.
point_y = {
    "brick:Temperature_Sensor": 330,
    "brick:Relative_Humidity_Sensor": 0,
    "brick:CO2_Sensor": -330,
}
for p in point_nodes:
    parent = linked_device(p)
    bt = point_brick_type(p)
    y = point_y.get(qname(bt), 0)

    if len(device_order) >= 1 and parent == device_order[0]:
        POS[p] = (-355, y)
    elif len(device_order) >= 2 and parent == device_order[1]:
        POS[p] = (355, y)

# Observations are intentionally offset both horizontally AND vertically.
# This prevents observation/property/point nodes from lying on one line.
OBS_OFFSET = {
    330: 90,
    0: 90,
    -330: 90,
}

for obs in obs_nodes:
    p = obs_point(obs)
    if p not in POS:
        continue

    px, py = POS[p]

    # Observation moves inward toward centre.
    ox = -155 if px < 0 else 155

    # Alternate vertical offset by side:
    # left observation above the property, right below.
    dy = OBS_OFFSET.get(py, 90)
    oy = py + dy if px < 0 else py - dy

    POS[obs] = (ox, oy)

# Ontology classes on the outer perimeter, away from devices/points.
for n in class_nodes:
    q = qname(n)

    if q == "ex:OperationalDevice":
        POS[n] = (0, 565)

    elif q == "sosa:Observation":
        POS[n] = (0, -565)

    elif q == "brick:Temperature_Sensor":
        POS[n] = (-650, 410)

    elif q == "brick:Relative_Humidity_Sensor":
        POS[n] = (-700, 110)

    elif q == "brick:CO2_Sensor":
        POS[n] = (-650, -410)

# Legend placed in free lower-right space.
A.get_node("LEGEND").attr["pos"] = "610,-500!"
A.get_node("LEGEND").attr["pin"] = "true"

# Apply pinned coordinates.
for node, (x, y) in POS.items():
    if node in gv_id:
        gvnode = A.get_node(gv_id[node])
        gvnode.attr["pos"] = f"{x},{y}!"
        gvnode.attr["pin"] = "true"

# Verify that every displayed RDF node received a controlled position.
unpositioned = [n for n in G.nodes() if n not in POS]
if unpositioned:
    print("\nWARNING: unpositioned RDF nodes:")
    for n in unpositioned:
        print(" -", qname(n), G.nodes[n]["type"])
else:
    print("✓ V9 layout: every displayed RDF node has a controlled position.")

# Respect supplied coordinates and route edges.
A.layout(prog="neato", args="-n2")

# ------------------------------------------------------------
# 13. EXPORT
# ------------------------------------------------------------

SVG_PATH = OUTPUT_DIR / "Figure_2b_RDF_V13_final.svg"
PDF_PATH = OUTPUT_DIR / "Figure_2b_RDF_V13_final.pdf"
PNG_PATH = OUTPUT_DIR / "Figure_2b_RDF_V13_final.png"

A.draw(str(SVG_PATH), format="svg")
A.draw(str(PDF_PATH), format="pdf")
A.draw(str(PNG_PATH), format="png")

# ------------------------------------------------------------
# 14. REPRODUCIBILITY AUDIT FILE
# ------------------------------------------------------------

AUDIT_PATH = OUTPUT_DIR / "Figure_2b_RDF_V13_displayed_triples.tsv"

with AUDIT_PATH.open("w", encoding="utf-8") as f:
    f.write("subject\tpredicate\tobject\n")
    for s, p, o in selected_triples:
        f.write(
            f"{str(s)}\t{str(p)}\t{str(o)}\n"
        )

print("\n==============================================")
print("FIGURE 2(b) V13 FINAL COMPLETED")
print("==============================================")
print(f"Displayed nodes   : {G.number_of_nodes()}")
print(f"Displayed triples : {G.number_of_edges()}")
print("\nFiles:")
print(" SVG :", SVG_PATH.resolve())
print(" PDF :", PDF_PATH.resolve())
print(" PNG :", PNG_PATH.resolve())
print(" AUDIT:", AUDIT_PATH.resolve())

print(
    "\nScientific guarantee: every displayed edge was copied "
    "from an explicit triple in the loaded TTL. "
    "No ex:hasSemanticPoint relation is assumed or generated."
)
